# 🚀 Nova-LLM × Llama 3.2 3B — Unified Reasoning & Live Web-Search LLM

Combines **Llama 3.2 3B Instruct** (real 3-billion-parameter general intelligence!) with your **`suniljaiswal39375-crypto/llm` (`nova_llm`)** stack:
- **Fine-tuning (T4 GPU, ~10 min):** LoRA (`r=16, alpha=32`) on **`yahma/alpaca-cleaned`** + **Multi-Turn Agentic Web-Search, Tool-Calling (`<|tool_call|>`), `<|think|>` Reasoning & Inline Citation (`[1]`, `[2]`)** traces.
- **Real Live Internet Search (FREE — Zero API Key Needed):** Uses `nova_llm`'s `HybridSearchBackend` with live **DuckDuckGo + Wikipedia** (or optional `TAVILY_API_KEY` / `BRAVE_API_KEY`) + full web page reading (`fetch_page`).
- **Bulletproof Colab Compatibility:** Verified `nova_llm` import check, automatic Unsloth → standard HF 4-bit PEFT fallback (Python 3.10–3.13 safe), resilient Google Drive mounting with local ZIP backup, and zero W&B prompts.
- **Built-in Live Browser Chat UI:** Launches `nova_llm`'s interactive Search-Chat UI directly from Colab.

**How to run:** `Runtime → Change runtime type → T4 GPU` → `Runtime → Run all`.

In [ ]:
# 1. Clone Nova-LLM repo, install dependencies, and VERIFY all imports
import importlib, os, subprocess, sys
import torch

assert torch.cuda.is_available(), "❌ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU, then re-run."
print(f"🖥️ GPU detected: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__} | Python {sys.version.split()[0]}")

REPO_DIR = "/content/nova-llm"
SRC_DIR  = f"{REPO_DIR}/src"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/suniljaiswal39375-crypto/llm.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Install core required packages (fail loudly if anything fails)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "sentencepiece", "protobuf", "datasets>=3.0.0", "huggingface_hub>=0.28.0",
    "transformers>=4.46.0", "accelerate>=0.34.0", "peft>=0.13.0", "bitsandbytes>=0.44.0",
    "fastapi", "uvicorn", "httpx", "safetensors", "pyyaml",
], check=True)

# Try installing Unsloth for 2x faster training (Cell 3 automatically falls back to native HF 4-bit PEFT if unavailable)
unsloth_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=False)
if unsloth_res.returncode != 0:
    print("ℹ️ Unsloth optional install skipped on this runtime — Cell 3 will use standard 4-bit QLoRA (transformers + peft + bitsandbytes).")

# Register nova_llm on sys.path immediately AND install editable package
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=False)
importlib.invalidate_caches()

# Hard verification: make sure every Nova-LLM symbol used by later cells imports cleanly right now!
import nova_llm
from nova_llm.agent import (
    SEARCH_SYSTEM_PROMPT,
    HybridSearchBackend,
    LiveHTTPWebBackend,
    SearchAgent,
    TransformersBackend,
)
from nova_llm.inference import create_app

print(f"✅ Nova-LLM v{nova_llm.__version__} verified and imported from {nova_llm.__file__}!")

In [ ]:
# 2. Mount Google Drive safely (with stale-mount cleanup + automatic local fallback)
import os, shutil
from google.colab import drive

MOUNTPOINT = '/content/drive'
DRIVE_MYDRIVE = f'{MOUNTPOINT}/MyDrive'
LOCAL_FALLBACK = '/content/saved_models'

def mount_drive_safe():
    if os.path.isdir(DRIVE_MYDRIVE):
        return True
    if os.path.isdir(MOUNTPOINT) and not os.path.ismount(MOUNTPOINT):
        try:
            if os.listdir(MOUNTPOINT):
                shutil.rmtree(MOUNTPOINT)
        except Exception:
            pass
    try:
        drive.mount(MOUNTPOINT, force_remount=True, timeout_ms=300000)
        return os.path.isdir(DRIVE_MYDRIVE)
    except Exception as e:
        print(f'⚠️ Google Drive mount failed: {e}')
        print('👉 Tip: Allow popups & third-party cookies for colab.research.google.com, OR click the 📁 -> Mount Drive icon in the left sidebar.')
        return False

if mount_drive_safe():
    DRIVE_ROOT = DRIVE_MYDRIVE
    print('✅ Google Drive mounted & writable:', DRIVE_ROOT)
else:
    DRIVE_ROOT = LOCAL_FALLBACK
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('📁 Falling back to local Colab storage so training can continue:', DRIVE_ROOT)

LORA_DIR = f'{DRIVE_ROOT}/nova-llama3.2-3b-websearch-lora'
print('📁 LoRA adapters will be saved to:', LORA_DIR)

In [ ]:
# 3. Load Llama 3.2 3B Instruct (4-bit) + attach LoRA (Unsloth if available, else native HF 4-bit + PEFT)
import torch

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
USE_UNSLOTH = False

try:
    from unsloth import FastLanguageModel, get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=TARGET_MODULES,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    USE_UNSLOTH = True
    print("🚀 Loaded Llama 3.2 3B Instruct (4-bit) with Unsloth LoRA (r=16, alpha=32)!")
except Exception as unsloth_err:
    print(f"ℹ️ Using standard Hugging Face 4-bit QLoRA backend (Unsloth unavailable: {type(unsloth_err).__name__}: {unsloth_err})")
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=TARGET_MODULES,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.config.use_cache = False
    print("🚀 Loaded Llama 3.2 3B Instruct (4-bit) with standard PEFT LoRA (r=16, alpha=32)!")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 4. Hybrid Training Dataset: Alpaca General Instructions + Nova-Search Tool-Calling & Citation Traces
import json, sys
if "/content/nova-llm/src" not in sys.path:
    sys.path.insert(0, "/content/nova-llm/src")

from datasets import Dataset, concatenate_datasets, load_dataset
from nova_llm.agent import SEARCH_SYSTEM_PROMPT

# (A) General instruction-following & reasoning from yahma/alpaca-cleaned
alpaca = load_dataset("yahma/alpaca-cleaned", split="train")

def format_alpaca(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n\n{inp}".strip() if inp else inst
        msgs = [
            {"role": "system", "content": SEARCH_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

alpaca_ds = alpaca.select(range(4000)).map(format_alpaca, batched=True, remove_columns=alpaca.column_names)

# (B) Multi-turn Agentic Web-Search, <|think|> Reasoning, Citation Grounding & Abstention traces
SEARCH_TRACES = [
    (
        "What is the latest stable version of Python and its main new features?",
        "I should search the web for the latest stable Python release and highlights.",
        "latest stable Python release features",
        "[1] Python Release Notes (python.org): Python 3.13/3.14 adds free-threaded CPython (PEP 703), experimental JIT compiler, and improved error tracebacks.",
        "Recent stable Python releases introduce an experimental free-threaded mode disabling the GIL (PEP 703), an experimental JIT compiler, and clearer interactive tracebacks [1].",
    ),
    (
        "Who won the most recent ICC Cricket World Cup?",
        "Sports winners change over time, so I must search for the latest ICC Cricket World Cup winner.",
        "most recent ICC Cricket World Cup winner",
        "[1] ICC Official (icc-cricket.com): Australia won the 2023 Men's ODI Cricket World Cup, defeating India in the final in Ahmedabad.",
        "Australia won the 2023 Men's ODI Cricket World Cup, defeating India in the final in Ahmedabad [1].",
    ),
    (
        "Compare PyTorch FSDP2 and DDP for multi-GPU training.",
        "Let me search for PyTorch FSDP2 official documentation to provide an accurate, cited comparison.",
        "PyTorch FSDP2 vs DDP DTensor fully_shard",
        "[1] PyTorch Docs (pytorch.org): DDP replicates full model weights per GPU, while FSDP2 uses DTensor per-parameter sharding via fully_shard() to reduce memory across GPUs.",
        "DDP keeps a full copy of model parameters on every GPU, whereas PyTorch FSDP2 shards parameters, gradients, and optimizer states per-parameter using DTensor (`fully_shard()`), enabling much larger models to fit in VRAM [1].",
    ),
    (
        "What is the current stock price of NVIDIA?",
        "Live stock prices change every minute, so I must emit a web_search tool call.",
        "NVIDIA NVDA stock price today",
        "[1] Market Data: Live quote unavailable in this snippet.",
        "I could not retrieve a verified real-time quote for NVIDIA from the search results, so I cannot confirm the current price without a live market feed [1].",
    ),
    (
        "Explain how MuonClip prevents attention logit explosions.",
        "Let me check the Kimi K2 / MuonClip paper for exact technical details.",
        "Kimi K2 MuonClip QK-Clip attention logit explosion",
        "[1] arXiv 2507.20534 (arxiv.org): MuonClip combines Muon orthogonalized momentum with per-head QK-Clip, rescaling query and key weights whenever max attention logits exceed threshold tau.",
        "MuonClip pairs Muon's Newton-Schulz orthogonalized updates with per-head QK-Clip, dynamically rescaling query and key projection weights whenever attention logits exceed a safety threshold tau [1].",
    ),
    (
        "What is the weather in Panchkula, India right now?",
        "Current weather requires a live web search.",
        "Panchkula Haryana India current weather",
        "[1] Weather Report: Clear skies, 24°C, light breeze in Panchkula, Haryana.",
        "Current conditions in Panchkula, Haryana show clear skies around 24°C with a light breeze [1].",
    ),
]

search_texts = []
for user_q, thought, query, snippet, cited_ans in SEARCH_TRACES * 40:
    tc = '<|tool_call|>' + json.dumps({"name": "web_search", "arguments": {"query": query}}) + '</|tool_call|>'
    msgs = [
        {"role": "system", "content": SEARCH_SYSTEM_PROMPT},
        {"role": "user", "content": user_q},
        {"role": "assistant", "content": f"<|think|>{thought}</|think|>\n{tc}"},
        {"role": "user", "content": f"Search Results:\n<|tool_result|>{snippet}</|tool_result|>\nNow answer the user's question concisely with inline citations like [1] and [2]."},
        {"role": "assistant", "content": cited_ans},
    ]
    search_texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))

search_ds = Dataset.from_dict({"text": search_texts})
train_ds = concatenate_datasets([alpaca_ds, search_ds]).shuffle(seed=3407)
print(f"✅ Hybrid training set ready: {len(train_ds):,} examples ({len(alpaca_ds)} Alpaca + {len(search_ds)} Web-Search traces)")

In [ ]:
# 5. Fine-tune Llama 3.2 3B on T4 GPU (cosine LR, 8-bit AdamW, report_to='none' so no W&B prompt)
import torch
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

trainer = Trainer(
    model=model,
    train_dataset=tokenized_ds,
    data_collator=collator,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=True,
    ),
)
stats = trainer.train()
print(f"✅ Fine-tuning finished in {stats.metrics['train_runtime']/60:.1f} min! Final loss: {stats.training_loss:.4f}")

In [ ]:
# 6. SAVE #1 — Save LoRA adapters & tokenizer (auto-detects Google Drive or saves verified local ZIP)
import os, shutil

if os.path.isdir('/content/drive/MyDrive'):
    DRIVE_ROOT = '/content/drive/MyDrive'
    LORA_DIR = f'{DRIVE_ROOT}/nova-llama3.2-3b-websearch-lora'
else:
    os.makedirs(DRIVE_ROOT, exist_ok=True)

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

files = sorted(os.listdir(LORA_DIR))
total = sum(os.path.getsize(os.path.join(LORA_DIR, f)) for f in files)
assert total > 1_000_000, "LoRA adapter size too small - re-run save!"
print(f"✅ Verified LoRA checkpoint saved to: {LORA_DIR} ({total/1e6:.1f} MB)")

if not LORA_DIR.startswith('/content/drive/'):
    zip_path = shutil.make_archive('/content/nova-llama3.2-3b-websearch-lora', 'zip', LORA_DIR)
    print(f"📦 Local backup ZIP created: {zip_path} — download from left Files sidebar or mount Drive & re-run Cell 6.")

In [ ]:
# 7. LIVE AGENTIC WEB SEARCH TEST — Real Llama 3.2 3B + FREE Live Internet (DuckDuckGo + Wikipedia)!
# Optional: set os.environ['TAVILY_API_KEY'] = 'tvly-...' for even richer AI web search.
import sys
if "/content/nova-llm/src" not in sys.path:
    sys.path.insert(0, "/content/nova-llm/src")

from nova_llm.agent import HybridSearchBackend, LiveHTTPWebBackend, SearchAgent, TransformersBackend

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()
    model.config.use_cache = True

backend = TransformersBackend(model=model, tokenizer=tokenizer, model_name="Nova-Llama-3.2-3B-WebSearch")
search_backend = HybridSearchBackend(live_backend=LiveHTTPWebBackend(enable_free_web=True))
agent = SearchAgent(backend=backend, search_backend=search_backend, max_search_rounds=2, top_k_results=3)

def ask_web(question, force_search=False):
    res = agent.run(question, max_tokens=256, temperature=0.2, force_search=force_search)
    if res.thinking:
        print("💭 Thinking:", res.thinking)
    if res.tool_calls:
        print("🔍 Web Search:", res.tool_calls)
    for s in res.sources:
        print(f"   [{s.id}] {s.title} ({s.url})")
    print("\n✨ Answer:\n" + res.answer)

# Try ANY real-world question!
ask_web("What are the main breakthroughs in PyTorch FSDP2?", force_search=True)

In [ ]:
# 8. Launch Live Interactive Nova-Search Browser Chat UI inside Google Colab!
import socket, sys, threading, time, uvicorn
if "/content/nova-llm/src" not in sys.path:
    sys.path.insert(0, "/content/nova-llm/src")

from google.colab import output
from nova_llm.inference import create_app

def _pick_free_port(start=8000):
    for port in range(start, start + 20):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return start

if not globals().get("_NOVA_SERVER_PORT"):
    _NOVA_SERVER_PORT = _pick_free_port(8000)
    app = create_app(backend=backend, model_name="Nova-Llama-3.2-3B-WebSearch", search_backend=search_backend)

    def _run():
        uvicorn.run(app, host="0.0.0.0", port=_NOVA_SERVER_PORT, log_level="warning")

    threading.Thread(target=_run, daemon=True).start()
    time.sleep(2)

print(f"🌐 Opening Nova-Search Web UI on port {_NOVA_SERVER_PORT} (powered by Llama 3.2 3B + live internet search)...")
output.serve_kernel_port_as_window(_NOVA_SERVER_PORT)

In [ ]:
# 9. OPTIONAL — Export quantized GGUF (q4_k_m, ~2 GB) for LM Studio / Ollama (or merged weights in PEFT mode)
import glob, os, shutil, time

target_dir = '/content/drive/MyDrive' if os.path.isdir('/content/drive/MyDrive') else DRIVE_ROOT

if USE_UNSLOTH and hasattr(model, "save_pretrained_gguf"):
    model.save_pretrained_gguf("nova_llama32_3b_gguf", tokenizer, quantization_method="q4_k_m")

    def copy_to_drive_verified(src, dst_dir, attempts=3):
        if not dst_dir.startswith('/content/drive/'):
            os.makedirs(dst_dir, exist_ok=True)
            return shutil.copy(src, dst_dir)
        for i in range(attempts):
            if not os.path.isdir(dst_dir):
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
                except Exception as e:
                    print(f"Drive remount skipped ({e}); keeping GGUF locally at {src}")
                    return src
            try:
                out = shutil.copy(src, dst_dir)
                if os.path.getsize(out) == os.path.getsize(src):
                    return out
            except Exception as e:
                print(f"Retry {i+1}/{attempts}: {e}")
            time.sleep(3)
        print(f"⚠️ Could not copy to Drive; GGUF is still saved locally at {src}")
        return src

    for f in glob.glob("nova_llama32_3b_gguf_gguf/*.gguf") + glob.glob("nova_llama32_3b_gguf/*.gguf"):
        dst = copy_to_drive_verified(f, target_dir)
        print(f"✅ GGUF verified at: {dst} ({os.path.getsize(dst)/1e9:.2f} GB)")
else:
    print("ℹ️ GGUF export requires Unsloth. Your verified LoRA checkpoint is already saved at:", LORA_DIR)
    print("👉 You can load it anywhere with: python scripts/serve.py --hf-model unsloth/Llama-3.2-3B-Instruct-bnb-4bit --lora-dir", LORA_DIR, "--live-web")